# Multifactorial Meta-Learning Benchmark for Small-Data Machine Learning in Soft Matter and Biomaterials Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This notebook follows a step-by-step workflow:
- load the dataset metadata from the Croissant file,
- inspect available record sets and their field `@id`s,
- load records into pandas DataFrames,
- perform a lightweight exploratory analysis using dynamically selected fields,
- create a few example visualizations.

> Important: all entities in the dataset should be referenced by their `@id` values. Because this dataset may expose record sets dynamically, the notebook first discovers them from metadata and then uses those discovered `@id`s in subsequent steps.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import json
from pprint import pprint

import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)
sns.set_theme(style='whitegrid')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.0bxf-db06/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Because Croissant datasets can include multiple record sets, we first inspect the metadata in a generic way. This notebook avoids assuming a fixed structure and instead discovers usable entities from the metadata object.

In [ ]:
def to_jsonable(obj):
    if hasattr(obj, 'to_json'):
        return obj.to_json()
    return obj

metadata_json = to_jsonable(metadata)
print(type(metadata_json))
print('Top-level keys:')
print(sorted(metadata_json.keys())[:50])

In [ ]:
# Discover record sets from metadata JSON using their @id values.
record_sets_meta = metadata_json.get('recordSet', []) or metadata_json.get('recordset', []) or []

print(f"Number of record sets declared in metadata: {len(record_sets_meta)}")

record_set_summaries = []
for rs in record_sets_meta:
    rs_id = rs.get('@id', 'N/A')
    rs_name = rs.get('name', '')
    fields = rs.get('field', [])
    field_ids = [f.get('@id', 'N/A') for f in fields]
    record_set_summaries.append({
        'record_set_id': rs_id,
        'record_set_name': rs_name,
        'n_fields': len(fields),
        'field_ids_preview': field_ids[:10]
    })

record_set_summary_df = pd.DataFrame(record_set_summaries)
record_set_summary_df

In [ ]:
# Show detailed field-level metadata for each record set, referencing fields by @id.
field_rows = []

for rs in record_sets_meta:
    rs_id = rs.get('@id', 'N/A')
    rs_name = rs.get('name', '')
    for field in rs.get('field', []):
        field_rows.append({
            'record_set_id': rs_id,
            'record_set_name': rs_name,
            'field_id': field.get('@id', 'N/A'),
            'field_name': field.get('name', ''),
            'data_type': field.get('dataType', ''),
            'description': field.get('description', ''),
            'source': field.get('source', {})
        })

field_df = pd.DataFrame(field_rows)
field_df.head(20)

The next cell prints sample records from the first available record set using its discovered `@id`.

If no record sets are declared in metadata, the notebook will gracefully report that and skip loading.

In [ ]:
record_set_ids = [rs.get('@id') for rs in record_sets_meta if rs.get('@id')]
print('Discovered record set @id values:')
print(record_set_ids)

if record_set_ids:
    sample_record_set_id = record_set_ids[0]
    print(f"\nSample records from record set: {sample_record_set_id}")
    try:
        for i, x in enumerate(dataset.records(record_set=sample_record_set_id)):
            print(x)
            if i >= 2:
                break
    except Exception as e:
        print(f"Unable to iterate records for {sample_record_set_id}: {e}")
else:
    print('No record sets were found in the Croissant metadata.')

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis. We use discovered record set and field `@id`s directly from metadata.

In [ ]:
# Extract data from each record set into a dictionary keyed by record set @id.
dataframes = {}
load_status = []

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        load_status.append({
            'record_set_id': record_set_id,
            'n_rows': len(df),
            'n_columns': len(df.columns),
            'status': 'loaded'
        })
    except Exception as e:
        load_status.append({
            'record_set_id': record_set_id,
            'n_rows': None,
            'n_columns': None,
            'status': f'failed: {e}'
        })

load_status_df = pd.DataFrame(load_status)
load_status_df

In [ ]:
# Preview loaded DataFrames.
if dataframes:
    example_record_set_id = next(iter(dataframes.keys()))
    print(f"Example record set @id: {example_record_set_id}")
    print('Columns:')
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print('No DataFrames were successfully loaded from record sets.')

To make the notebook reusable, we now automatically choose:
- one loaded record set,
- one numeric field `@id`,
- one grouping field `@id` when available.

This mirrors a common exploratory workflow while keeping the code robust to schema changes.

In [ ]:
selected_record_set_id = None
selected_numeric_field_id = None
selected_group_field_id = None

for rs_id, df in dataframes.items():
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    candidate_group_cols = [c for c in df.columns if df[c].dtype == 'object' or str(df[c].dtype).startswith('category')]
    if len(df) > 0 and numeric_cols:
        selected_record_set_id = rs_id
        selected_numeric_field_id = numeric_cols[0]
        selected_group_field_id = candidate_group_cols[0] if candidate_group_cols else None
        break

print(f"Selected record set @id: {selected_record_set_id}")
print(f"Selected numeric field @id: {selected_numeric_field_id}")
print(f"Selected group field @id: {selected_group_field_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section uses dynamically discovered field `@id`s. If the dataset does not contain numeric columns in loaded record sets, the notebook will skip these steps gracefully.

In [ ]:
if selected_record_set_id is not None and selected_numeric_field_id is not None:
    analysis_df = dataframes[selected_record_set_id].copy()
    numeric_field = selected_numeric_field_id

    clean_series = pd.to_numeric(analysis_df[numeric_field], errors='coerce').dropna()
    if len(clean_series) > 0:
        threshold = clean_series.median()
        filtered_df = analysis_df[pd.to_numeric(analysis_df[numeric_field], errors='coerce') > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        std = filtered_df[numeric_field].std()
        if pd.notna(std) and std != 0:
            filtered_df[f"{numeric_field}_normalized"] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std
            )
        else:
            filtered_df[f"{numeric_field}_normalized"] = 0.0

        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        group_field = selected_group_field_id
        if group_field is not None and group_field in filtered_df.columns:
            numeric_subset = filtered_df.select_dtypes(include=[np.number])
            if not numeric_subset.empty:
                grouped_df = filtered_df.groupby(group_field)[numeric_subset.columns].mean(numeric_only=True)
                print(f"Grouped data by {group_field}:")
                display(grouped_df.head())
            else:
                print('No numeric columns available for grouped aggregation.')
        else:
            print('No suitable grouping field was found for this record set.')
    else:
        print('The selected numeric field does not contain valid numeric values after coercion.')
else:
    print('EDA skipped because no loaded record set with numeric fields was found.')

In [ ]:
# Optional additional profiling: missingness and basic summary statistics.
if selected_record_set_id is not None:
    profiling_df = dataframes[selected_record_set_id].copy()
    print(f"Basic profiling for record set @id: {selected_record_set_id}")
    missing_df = pd.DataFrame({
        'field_id': profiling_df.columns,
        'missing_count': profiling_df.isna().sum().values,
        'missing_fraction': profiling_df.isna().mean().values
    }).sort_values('missing_fraction', ascending=False)
    display(missing_df.head(20))

    numeric_profile = profiling_df.select_dtypes(include=[np.number]).describe().T
    display(numeric_profile.head(20))
else:
    print('Profiling skipped because no record set was loaded.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The plots below are generated only when appropriate numeric fields are available.

In [ ]:
if selected_record_set_id is not None and selected_numeric_field_id is not None:
    viz_df = dataframes[selected_record_set_id].copy()
    num_col = selected_numeric_field_id
    viz_df[num_col] = pd.to_numeric(viz_df[num_col], errors='coerce')
    viz_df = viz_df.dropna(subset=[num_col])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.histplot(viz_df[num_col], kde=True, ax=axes[0], color='steelblue')
    axes[0].set_title(f'Distribution of {num_col}')
    axes[0].set_xlabel(num_col)

    sns.boxplot(x=viz_df[num_col], ax=axes[1], color='salmon')
    axes[1].set_title(f'Boxplot of {num_col}')
    axes[1].set_xlabel(num_col)

    plt.tight_layout()
    plt.show()
else:
    print('Visualization skipped because no suitable numeric field was found.')

In [ ]:
# If at least two numeric fields exist, visualize their relationship.
if selected_record_set_id is not None:
    rel_df = dataframes[selected_record_set_id].copy()
    numeric_cols = rel_df.select_dtypes(include=[np.number]).columns.tolist()

    if len(numeric_cols) >= 2:
        x_field_id = numeric_cols[0]
        y_field_id = numeric_cols[1]
        plot_df = rel_df[[x_field_id, y_field_id]].dropna().copy()

        plt.figure(figsize=(7, 5))
        sns.scatterplot(data=plot_df, x=x_field_id, y=y_field_id, alpha=0.7)
        plt.title(f'{y_field_id} vs {x_field_id}')
        plt.xlabel(x_field_id)
        plt.ylabel(y_field_id)
        plt.tight_layout()
        plt.show()
    else:
        print('Relationship plot skipped because fewer than two numeric fields were available.')
else:
    print('Relationship plot skipped because no record set was loaded.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:

- loaded a Croissant dataset using `mlcroissant`,
- accessed dataset metadata as a single object,
- discovered available record sets and fields using their `@id` values,
- loaded record sets into pandas DataFrames,
- performed a basic exploratory workflow with filtering, normalization, grouping, profiling, and visualization.

For this FAIR^2 dataset, the most important next step is to inspect the discovered record set `@id`s and choose the specific benchmark tables most relevant to your analysis, such as split-level metrics, aggregated metrics, or meta-feature tables. Once identified, you can extend this notebook to:

- compare algorithm performance across tasks,
- analyze variability across random seeds or folds,
- study the relationship between dataset meta-features and model outcomes,
- build reproducible benchmarking and reporting pipelines.

Because all downstream references use record set and field `@id`s, the workflow remains aligned with Croissant metadata best practices.